# NeuroFusionAI: Step 3 - Explainable AI (XAI) Dashboard
This notebook visualizes and analyzes model predictions using:
1. **Grad-CAM** (heatmaps for CNN drawings classifier) to show drawing strokes contributing to prediction.
2. **SHAP summary and bar plots** for the voice and telemonitoring models to rank acoustic and physiological feature importances.


In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

sys.path.append(os.path.abspath('..'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 1. Grad-CAM on Drawings Model
We implement Grad-CAM (Gradient-weighted Class Activation Mapping) on the final convolution layer of our ResNet-18 model.


In [ ]:
from models.image_model import ImageDrawingClassifier
from preprocessing.image_preprocessing import get_image_dataloader

# Load trained image model
image_model = ImageDrawingClassifier(num_classes=2).to(device)
if os.path.exists('outputs/checkpoints/image_best.pth'):
    image_model.load_state_dict(torch.load('outputs/checkpoints/image_best.pth', map_location=device))
image_model.eval()

# Grad-CAM hooks
gradients = []
activations = []

def save_gradient(grad):
    gradients.append(grad)

def forward_hook(module, input, output):
    activations.append(output)
    output.register_hook(save_gradient)

# Hook the last conv block of ResNet-18 (resnet.layer4[1].conv2)
target_layer = image_model.resnet.layer4[1].conv2
target_layer.register_forward_hook(forward_hook)

# Get one validation drawing
val_loader = get_image_dataloader('validation', batch_size=1, shuffle=True)
imgs, labels = next(iter(val_loader))
imgs, labels = imgs.to(device), labels.to(device)
imgs.requires_grad = True

# Forward pass
output = image_model(imgs)
_, pred = output.max(1)

# Backward pass to extract gradients
loss = output[0, pred.item()]
loss.backward()

# Generate Grad-CAM heatmap
acts = activations[0].cpu().detach().numpy()[0]  # Shape: (512, 7, 7)
grads = gradients[0].cpu().detach().numpy()[0]  # Target activation gradients
weights = np.mean(grads, axis=(1, 2))  # Mean gradients

cam = np.zeros(acts.shape[1:], dtype=np.float32)
for i, w in enumerate(weights):
    cam += w * acts[i, :, :]
    
cam = np.maximum(cam, 0)  # ReLU
if cam.max() > 0:
    cam = cam / cam.max()
cam = cv2.resize(cam, (224, 224))

# Plot drawing overlay
img_np = imgs[0].cpu().detach().numpy().transpose(1, 2, 0)
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
heatmap = np.float32(heatmap) / 255.0
overlay = heatmap * 0.4 + img_np
overlay = overlay / overlay.max()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_np)
axes[0].set_title('Original Drawing')
axes[0].axis('off')
axes[1].imshow(overlay)
axes[1].set_title('Grad-CAM Heatmap overlay')
axes[1].axis('off')
plt.savefig('outputs/plots/gradcam_visual.png')
plt.show()


## 2. Feature Attribution with SHAP on Voice Classification Model
We use SHAP (SHapley Additive exPlanations) to explain features on the tabular XGBoost voice classification model.


In [ ]:
import shap
from models.voice_model import VoiceXGBClassifier

xgb_model = VoiceXGBClassifier()
if os.path.exists('outputs/checkpoints/voice_xgb_best.pkl'):
    xgb_model.load('outputs/checkpoints/voice_xgb_best.pkl')
    
val_df = pd.read_csv('datasets/validation/voice/oxford_validation.csv')
X_val = val_df.drop(columns=['status']).values
feature_names = val_df.drop(columns=['status']).columns.tolist()

explainer = shap.TreeExplainer(xgb_model.model)
shap_values = explainer.shap_values(X_val)

plt.figure()
shap.summary_plot(shap_values, X_val, feature_names=feature_names, show=False)
plt.title('SHAP Feature Importance for Voice Classification')
plt.tight_layout()
plt.savefig('outputs/plots/shap_voice_summary.png')
plt.show()
